# 07 · Phase 12 (part 2) — Deep reward oracle: a weight-sharing supernet

The classical oracle of Notebook 06 is fast but is a *different classifier* from the network
that is finally deployed. Its preferences (e.g. very short windows, which suit hand-crafted
floor-to-peak statistics) need not transfer to a CNN. The reward should instead come from the
same network family that the policy will drive, as in RLWBS, where the backbone is trained on the
wavelet features the policy produces.

**Supernet.** One tri-branch fusion network is trained with a *random configuration for every
training crop* (random window per batch; random wavelet, level, band and branch subset per item).
Its weights are shared by all 2,100 configurations, which makes "how good is configuration c for
recording i?" a single forward pass (one-shot / weight-sharing evaluation).

* **Out-of-fold**: 3 supernets, each trained on 2/3 of the training patients, score the remaining
  third → training-split rewards never come from a network that saw the recording.
* A 4th supernet trained on the whole training split scores validation and test recordings and is
  the initialisation for Phase-13 fine-tuning (Notebook 16).
* Scoring is factorised: T features depend only on the window, S on (band, window), W on
  (wavelet, level, band, window); the 7 fusion masks are applied afterwards. The central 4 s of
  each recording is tiled by non-overlapping windows of the configuration's length.
* Clean and 10 dB pink-noise versions are scored (robustness reward term).

In [ ]:
%run common/00_config.ipynb
%run common/01_signal_lib.ipynb
%run common/02_rl_lib.ipynb
%run common/03_deep_lib.ipynb
import zlib
import matplotlib.pyplot as plt
import torch
from scipy.stats import spearmanr
from sklearn.model_selection import StratifiedGroupKFold

device = torch.device("cuda")
space = ConfigSpace(CFG)
meta, signals = load_deep_data(P)
fs = CFG["preprocessing"]["fs"]
y = meta.y.values.astype(np.float32)
idx = {s: np.where(meta.split == s)[0] for s in ("train", "val", "test")}
base = config_dict(space, space.baseline)
store = RecordStore(signals, [base] * len(meta), fs, CFG["deep"])
noisy = [add_noise(s, CFG["reward"]["robustness_snr_db"], np.random.default_rng(zlib.crc32(r.encode())), CFG["reward"]["robustness_noise"], fs)
         for s, r in zip(signals, meta.record_id)]
SN = dict(CFG["deep"], epochs=40, patience=10)
out_dir = P["models"] / "supernet"
out_dir.mkdir(parents=True, exist_ok=True)


def train_supernet(name, tr):
    path = out_dir / f"{name}.pt"
    model = FusionNet()
    if path.exists():
        model.load_state_dict(torch.load(path, map_location="cpu"))
        return model.to(device), pd.read_csv(out_dir / f"{name}_history.csv")
    t0 = time.time()
    model, hist = fit(model, store, y, tr, idx["val"], SN, device, seed=CFG["seed"], space=space, log=lambda s: None)
    torch.save(model.state_dict(), path)
    hist = pd.DataFrame(hist)
    hist.to_csv(out_dir / f"{name}_history.csv", index=False)
    print(f"{name}: {len(hist)} epochs, best val AUROC (baseline config) {hist.val_auroc.max():.3f}, {(time.time() - t0) / 60:.1f} min")
    return model, hist

In [ ]:
probs = {s: {k: np.zeros((space.n, len(idx[s])), np.float32) for k in ("clean", "noisy")} for s in idx}
folds = list(StratifiedGroupKFold(3, shuffle=True, random_state=CFG["seed"]).split(idx["train"], y[idx["train"]], meta.patient_id.values[idx["train"]]))
hists = {}
for f, (a, b) in enumerate(folds):
    model, hists[f"fold{f}"] = train_supernet(f"supernet_fold{f}", idx["train"][a])
    held = idx["train"][b]
    t0 = time.time()
    probs["train"]["clean"][:, b] = supernet_config_probs(model, [signals[i] for i in held], space, CFG, device)
    probs["train"]["noisy"][:, b] = supernet_config_probs(model, [noisy[i] for i in held], space, CFG, device)
    print(f"  fold {f}: scored {len(held)} held-out recordings x {space.n} configs x 2 in {(time.time() - t0) / 60:.1f} min")
    del model
    torch.cuda.empty_cache()
model, hists["full"] = train_supernet("supernet_full", idx["train"])
torch.save(model.state_dict(), P["models"] / "supernet" / "model_s0.pt")
for s in ("val", "test"):
    probs[s]["clean"] = supernet_config_probs(model, [signals[i] for i in idx[s]], space, CFG, device)
    probs[s]["noisy"] = supernet_config_probs(model, [noisy[i] for i in idx[s]], space, CFG, device)
np.savez_compressed(P["processed"] / "deep_probs.npz", **{f"{s}_{k}": probs[s][k] for s in probs for k in probs[s]})

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
for k, h in hists.items():
    ax.plot(h.epoch, h.val_auroc, label=k)
ax.set_xlabel("epoch"); ax.set_ylabel("val AUROC (baseline configuration)"); ax.legend(); ax.set_title("supernet training")
plt.tight_layout(); savefig(fig, "07_supernet_training"); plt.show()

## What does the deep oracle prefer, and does it agree with the classical oracle?

In [ ]:
prox = np.load(P["processed"] / "proxy_probs.npz")
rows = []
for c in space.all_configs():
    md = binary_metrics(y[idx["train"]], probs["train"]["clean"][c])
    mp = binary_metrics(y[idx["train"]], prox["train_clean"][c])
    rows.append({"config": c, **{k: str(v) for k, v in space.values(c).items()}, "deep_oof_auroc": md["auroc"], "deep_oof_bal_acc": md["balanced_acc"],
                 "classical_oof_auroc": mp["auroc"], "classical_oof_bal_acc": mp["balanced_acc"]})
ct = pd.DataFrame(rows)
ct.to_csv(P["tables"] / "07_oracle_config_scores.csv", index=False)
rho = spearmanr(ct.deep_oof_auroc, ct.classical_oof_auroc).correlation
print(f"Spearman rank correlation of configuration AUROC, deep vs classical oracle: {rho:.3f}")
print(f"baseline config OOF AUROC: deep {ct.deep_oof_auroc[space.baseline]:.3f} | classical {ct.classical_oof_auroc[space.baseline]:.3f}")
print(f"best config (deep oracle): {space.label(int(ct.deep_oof_auroc.idxmax()))} {ct.deep_oof_auroc.max():.3f}")
print(f"best config (classical oracle): {space.label(int(ct.classical_oof_auroc.idxmax()))} {ct.classical_oof_auroc.max():.3f}")

fig, ax = plt.subplots(1, 6, figsize=(28, 4))
ax[0].scatter(ct.classical_oof_auroc, ct.deep_oof_auroc, s=3, alpha=0.4); ax[0].set_xlabel("classical oracle AUROC"); ax[0].set_ylabel("deep oracle AUROC")
ax[0].set_title(f"per-configuration agreement (ρ = {rho:.2f})")
for a, comp in zip(ax[1:], ["wavelet", "level", "band", "window_s", "fusion"]):
    g = ct.groupby(comp)[["deep_oof_auroc", "classical_oof_auroc"]].mean()
    g.plot.bar(ax=a, legend=a is ax[1]); a.set_ylim(0.5, 1.0); a.set_title(f"mean OOF AUROC by {comp}"); a.tick_params(axis="x", rotation=30)
plt.tight_layout(); savefig(fig, "07_oracle_comparison"); plt.show()
ct.sort_values("deep_oof_auroc", ascending=False).head(10)